### Testing RAG Applications - (Advanced ⚡️) 📑

#### RAG Application
This application reads data about Model Context Protocol (MCP) server from internet, stores in vector stores, chunks the data with embedding and useful to answer the question about MCP while inferenced.

<img src="./img/RAG.png" width="500" height="400" style="display: block; margin: auto;">

In [31]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [32]:
# from langchain_ollama import OllamaEmbeddings
from langchain_groq import ChatGroq
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from typing import List
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document
# from langchain_ollama import ChatOllama

### If you are using Ollama Model within RAG then perform below step

In [3]:
# llm = ChatOllama(
#     base_url="http://localhost:11434",
#     model = "qwen2.5:latest",
#     temperature=0.5,
#     max_tokens = 250
# )

### If you are using Groq API Model within RAG then perform below step

In [33]:
# Initialize Groq LLM
llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0.5
)

In [34]:
llm.invoke([{"role": "user", "content": "Who is the president of USA in 2025, just give me the name"}])

AIMessage(content='Joe Biden', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 3, 'prompt_tokens': 51, 'total_tokens': 54, 'completion_time': 0.010140084, 'prompt_time': 0.002438743, 'queue_time': 0.058312723, 'total_time': 0.012578827}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_6507bcfb6f', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--c4197599-f26f-4ce1-94ea-fd74ab2328cb-0', usage_metadata={'input_tokens': 51, 'output_tokens': 3, 'total_tokens': 54})

In [4]:
# Load data from Web
loader = WebBaseLoader("https://www.descope.com/learn/post/mcp")
data = loader.load()

# Split text into documents
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splits = text_splitter.split_documents(data)

# Add text to vector db
# embedding = OllamaEmbeddings(model="nomic-embed-text:latest")
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectordb = Chroma.from_documents(documents=splits, embedding=embedding)

# Create a retriever
retriever = vectordb.as_retriever()

def format_docs(docs: List[Document]) -> str:
    return "\n\n".join([d.page_content for d in docs])


template = """Answer the question based only on the following context:

    {context}
    
    Give a summary not the full detail

    Question: {question}
    """
prompt = ChatPromptTemplate.from_template(template)


def retrieve_and_format(question):
    # docs = retriever.get_relevant_documents(question)
    docs = retriever.invoke(question)
    return format_docs(docs)

chain = {"context": retrieve_and_format, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser()


C:\Users\Admin\AppData\Local\Temp\ipykernel_3224\3881388503.py:11: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


#### Output of the LLM Application

In [5]:
response = chain.invoke("What is MCP")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


In [6]:
print(response)

MCP (Model Context Protocol) is a protocol that enables communication between clients and servers, allowing AI applications to access specific functions and capabilities, with a focus on integrating various services and data sources.


### Testing RAG Application with DeepEval
<img src="./img/RAGTesting.png" width="800" height="400" style="display: block; margin: auto;">

In [ ]:
### If you are not using Ollama Provider for Deepeval then run the below command to unset Ollama


# !deepeval unset-ollama


🙌 OpenAI will still be used by default because OPENAI_API_KEY is set.


In [ ]:
### If you want to unset the local model configuration for Deepeval then run the below command

# !deepeval unset-local-model

🙌 OpenAI will still be used by default because OPENAI_API_KEY is set.


In [7]:
### Login to Confident AI 

import deepeval
deepeval.login(os.getenv("DEEPEVAL_API_KEY"))

🎉🥳 Congratulations! You've successfully logged in! 🙌

In [ ]:
### if you want to use Groq API as you LLM provider for Deepeval then please perform the below configuation

import os
os.environ["OPENAI_BASE_URL"] = ""  # Groq’s OpenAI-compatible API
os.environ["OPENAI_API_KEY"] = ""               # use your Groq key here

In [ ]:
### Set Groq model as local model in Deepeval

!deepeval set-local-model --model-name="openai/gpt-oss-20b" --base-url="" --api-key=""

Settings updated for this session. To persist, use --save=dotenv[:path] 
(default .env.local) or set DEEPEVAL_DEFAULT_SAVE=dotenv:.env.local
🙌 Congratulations! You're now using a local model `openai/gpt-oss-20b` for all
evals that require an LLM.


In [ ]:
### If you want to use Meta Llama Scout model via Groq API as your LLM provider for Deepeval then please perform the below configuration

# !deepeval set-local-model --model-name="meta-llama/llama-4-scout-17b-16e-instruct" --base-url="" --api-key=""

Settings updated for this session. To persist, use --save=dotenv[:path] 
(default .env.local) or set DEEPEVAL_DEFAULT_SAVE=dotenv:.env.local
🙌 Congratulations! You're now using a local model 
`meta-llama/llama-4-scout-17b-16e-instruct` for all evals that require an LLM.


In [8]:
### Creating Multiple Test Data for Evaluation

test_data = [
    {
        "input": "What is MCP",
        "expected_output": "The Model Context Protocol (MCP) addresses this challenge by providing a standardized way for LLMs to connect with external data sources and tools—essentially a “universal remote” for AI apps. Released by Anthropic as an open-source protocol, MCP builds on existing function calling by eliminating the need for custom integration between LLMs and other apps."
    },
    {
        "input": "What is Relationship between function calling & Model Context Protocol",
        "expected_output": "The Model Context Protocol (MCP) builds on top of function calling, a well-established feature that allows large language models (LLMs) to invoke predetermined functions based on user requests. MCP simplifies and standardizes the development process by connecting AI applications to context while leveraging function calling to make API interactions more consistent across different applications and model vendors."
    },
    {
        "input": "What are the core components of MCP, just give the heading",
        "expected_output":""" 
                    - MCP Client
                    - MCP Servers
                    - Protocol Handshake
                    - Capability Discovery
                """
    }
]

### Creating Goldens

In [9]:
### Create Evaluation Dataset

from deepeval.dataset import Golden, EvaluationDataset

goldens = []

for data in test_data:
    golden = Golden(
        input=data['input'],
        expected_output=data['expected_output']
    )
    
    goldens.append(golden)
    

print(goldens)

[Golden(input='What is MCP', actual_output=None, expected_output='The Model Context Protocol (MCP) addresses this challenge by providing a standardized way for LLMs to connect with external data sources and tools—essentially a “universal remote” for AI apps. Released by Anthropic as an open-source protocol, MCP builds on existing function calling by eliminating the need for custom integration between LLMs and other apps.', context=None, retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None), Golden(input='What is Relationship between function calling & Model Context Protocol', actual_output=None, expected_output='The Model Context Protocol (MCP) builds on top of function calling, a well-established feature that allows large language models (LLMs) to invoke predetermined functions based on user requests. MCP simplifies and standardizes the development process by connecting AI ap

In [10]:
for i in range(len(goldens)):
    golden = goldens[i]
    print(f"Test Data {i+1}:")
    print("Input:", golden.input)
    print("Expected Output:", golden.expected_output)
    print()

Test Data 1:
Input: What is MCP
Expected Output: The Model Context Protocol (MCP) addresses this challenge by providing a standardized way for LLMs to connect with external data sources and tools—essentially a “universal remote” for AI apps. Released by Anthropic as an open-source protocol, MCP builds on existing function calling by eliminating the need for custom integration between LLMs and other apps.

Test Data 2:
Input: What is Relationship between function calling & Model Context Protocol
Expected Output: The Model Context Protocol (MCP) builds on top of function calling, a well-established feature that allows large language models (LLMs) to invoke predetermined functions based on user requests. MCP simplifies and standardizes the development process by connecting AI applications to context while leveraging function calling to make API interactions more consistent across different applications and model vendors.

Test Data 3:
Input: What are the core components of MCP, just give 

In [11]:
### Create Evaluation Dataset

dataset = EvaluationDataset(goldens=goldens)

In [12]:
dataset

EvaluationDataset(test_cases=[], goldens=[Golden(input='What is MCP', actual_output=None, expected_output='The Model Context Protocol (MCP) addresses this challenge by providing a standardized way for LLMs to connect with external data sources and tools—essentially a “universal remote” for AI apps. Released by Anthropic as an open-source protocol, MCP builds on existing function calling by eliminating the need for custom integration between LLMs and other apps.', context=None, retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None), Golden(input='What is Relationship between function calling & Model Context Protocol', actual_output=None, expected_output='The Model Context Protocol (MCP) builds on top of function calling, a well-established feature that allows large language models (LLMs) to invoke predetermined functions based on user requests. MCP simplifies and standardizes th

In [14]:
### Since we have created the Evaluation Dataset, let's view the Test Cases. Currently, test cases are empty

dataset.test_cases

[]

In [13]:
dataset.goldens

[Golden(input='What is MCP', actual_output=None, expected_output='The Model Context Protocol (MCP) addresses this challenge by providing a standardized way for LLMs to connect with external data sources and tools—essentially a “universal remote” for AI apps. Released by Anthropic as an open-source protocol, MCP builds on existing function calling by eliminating the need for custom integration between LLMs and other apps.', context=None, retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None),
 Golden(input='What is Relationship between function calling & Model Context Protocol', actual_output=None, expected_output='The Model Context Protocol (MCP) builds on top of function calling, a well-established feature that allows large language models (LLMs) to invoke predetermined functions based on user requests. MCP simplifies and standardizes the development process by connecting AI a

In [16]:
### Push the Evaluation Dataset to Deepeval Platform (Confident AI)

dataset.push("test")

✅ Dataset successfully pushed to Confident AI! View at 
]8;id=177372;https://app.confident-ai.com/project/cmhdvo6st07abnu0gdsqju0yp/datasets/cmhhgf7d0001gni0g9tusg8as\https://app.confident-ai.com/project/cmhdvo6st07abnu0gdsqju0yp/datasets/cmhhgf7d0001gni0g9tusg8as]8;;\

In [18]:
### Pull the Evaluation Dataset from Deepeval Platform (Confident AI)

dataset.pull(alias="test")

c:\Users\Admin\Documents\GEN_AI\LLM_Evaluation\Test_AI\Dev\.venv\Lib\site-packages\rich\live.py:256: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

In [19]:
### As we can see the Evaluation Dataset has been pulled successfully and test cases are not populated

dataset

EvaluationDataset(test_cases=[], goldens=[Golden(input='What is MCP', actual_output=None, expected_output='The Model Context Protocol (MCP) addresses this challenge by providing a standardized way for LLMs to connect with external data sources and tools—essentially a “universal remote” for AI apps. Released by Anthropic as an open-source protocol, MCP builds on existing function calling by eliminating the need for custom integration between LLMs and other apps.', context=None, retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None), Golden(input='What is Relationship between function calling & Model Context Protocol', actual_output=None, expected_output='The Model Context Protocol (MCP) builds on top of function calling, a well-established feature that allows large language models (LLMs) to invoke predetermined functions based on user requests. MCP simplifies and standardizes th

In [21]:
### To populate the test cases in the Evaluation Dataset, we can loop through the goldens and create test cases

from deepeval.test_case import LLMTestCase

for golden in dataset.goldens:
    test_case = LLMTestCase(
        input=golden.input,                     ### User Input
        expected_output=golden.expected_output,  ### Ground Truth
        actual_output=chain.invoke(golden.input),   ### LLM Output
        retrieval_context=[retrieve_and_format(golden.input)]  ### Retriever Output
    )

    dataset.add_test_case(test_case)

In [22]:
### Test cases are now populated in the Evaluation Dataset with actual outputs and retrieval contexts

dataset.test_cases

[LLMTestCase(input='What is MCP', actual_output='MCP (Model Context Protocol) is a protocol that enables communication between AI applications and servers, allowing for the exchange of data and functionality. It provides a standard for clients and servers to interact, using JSON-RPC 2.0 as the underlying message standard, and supports various transport methods, including STDIO and HTTP+SSE.', expected_output='The Model Context Protocol (MCP) addresses this challenge by providing a standardized way for LLMs to connect with external data sources and tools—essentially a “universal remote” for AI apps. Released by Anthropic as an open-source protocol, MCP builds on existing function calling by eliminating the need for custom integration between LLMs and other apps.', context=None, retrieval_context=['affair. This dramatically lowers the barrier for developer adoption, especially among those already using AI-enabled tools. However, consumer-facing applications like Claude Desktop still requ

In [30]:
for i in range(len(dataset.test_cases)):
    print(f"Test Case {i+1}:")
    test_case = dataset.test_cases[i]
    print("Input:", test_case.input)
    print("Expected Output:", test_case.expected_output)
    print("Actual Output:", test_case.actual_output)
    print("Retrieval Context:", test_case.retrieval_context)
    print()

Test Case 1:
Input: What is MCP
Expected Output: The Model Context Protocol (MCP) addresses this challenge by providing a standardized way for LLMs to connect with external data sources and tools—essentially a “universal remote” for AI apps. Released by Anthropic as an open-source protocol, MCP builds on existing function calling by eliminating the need for custom integration between LLMs and other apps.
Actual Output: MCP (Model Context Protocol) is a protocol that enables communication between AI applications and servers, allowing for the exchange of data and functionality. It provides a standard for clients and servers to interact, using JSON-RPC 2.0 as the underlying message standard, and supports various transport methods, including STDIO and HTTP+SSE.
Retrieval Context: ['affair. This dramatically lowers the barrier for developer adoption, especially among those already using AI-enabled tools. However, consumer-facing applications like Claude Desktop still require manual configur

### Creating LLMTestCase with Goldens

In [27]:
import deepeval.metrics


deepeval.evaluate(
    dataset.test_cases, 
    metrics= [
        deepeval.metrics.AnswerRelevancyMetric(),
        deepeval.metrics.FaithfulnessMetric(),
        deepeval.metrics.ContextualPrecisionMetric(),
        deepeval.metrics.ContextualRelevancyMetric()
    ]
)

✨ You're running DeepEval's latest Answer Relevancy Metric! (using openai/gpt-oss-20b (Local Model), strict=False,
async_mode=True)...

✨ You're running DeepEval's latest Faithfulness Metric! (using openai/gpt-oss-20b (Local Model), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Precision Metric! (using openai/gpt-oss-20b (Local Model), 
strict=False, async_mode=True)...

✨ You're running DeepEval's latest Contextual Relevancy Metric! (using openai/gpt-oss-20b (Local Model), 
strict=False, async_mode=True)...

RetryError: RetryError[<Future at 0x1f986300490 state=finished raised RateLimitError>]